# Lesson 04 - Патерн Дизайну Використання Інструментів

У цьому уроці ви навчитесь патерну дизайну **Використання Інструментів** для AI агентів, використовуючи Microsoft Agent Framework (Python). Ми розглянемо:

- Визначення функціональних інструментів за допомогою декоратора `@tool` і типізованих параметрів
- Надання схем інструментів, щоб модель знала, що кожен інструмент робить
- Контроль виконання інструментів за допомогою `approval_mode`
- Повернення **структурованого виводу** через моделі Pydantic і `response_format`

Сценарій — це **агент бронювання подорожей**, який може шукати напрямки, перевіряти доступність і отримувати інформацію про рейси.


## Налаштування


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Визначення інструментів за допомогою декоратора @tool

Декоратор `@tool` перетворює звичайну функцію Python на інструмент, який агент може викликати.  
Ключові моменти:

- **Докстрінг** стає описом інструмента, який бачить модель.  
- **Анотації типів** (включно з `Annotated` із описами) визначають схему інструмента.  
- Параметр `approval_mode` контролює, чи повинен користувач схвалювати кожен виклик перед його виконанням.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Створення агента з кількома інструментами

Передайте всі три інструменти клієнту, щоб модель могла викликати будь-які з них, які їй потрібні для відповіді на запитання користувача.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Структурований вивід за допомогою інструментів

Встановивши `response_format` у модель Pydantic, агент змушений повернути добре типізований JSON-об'єкт замість вільного тексту. Це корисно, коли код, що виконується далі, потребує програмного використання результату.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Шаблони схвалення інструментів

Параметр `approval_mode` у `@tool` контролює, чи потрібне схвалення людини перед виконанням викликів інструменту:

| Режим | Поведінка |
|---|---|
| `"never_require"` | Інструмент запускається автоматично — підтвердження користувача не потрібне. |
| `"always_require"` | Кожен виклик має бути схвалений користувачем перед виконанням. |

Використовуйте `"always_require"` для інструментів, які мають побічні ефекти (наприклад, бронювання рейсу, списання коштів з кредитної картки), щоб людина залишалася в процесі.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Підсумок

У цьому уроці ви дізналися, як:

1. **Визначати інструменти** за допомогою декоратора `@tool` з типізованими параметрами та докстрінгами, які слугують схемою інструменту.
2. **Комбінувати кілька інструментів**, щоб агент міг викликати їх послідовно для відповіді на складні запити.
3. **Повернути структуруваний результат**, передаючи модель Pydantic як `response_format`.
4. **Контролювати затвердження інструменту** за допомогою `approval_mode`, щоб утримувати людину у циклі для чутливих операцій.

Ці патерни формують основу для створення надійних, готових до виробництва агентів, які можуть безпечно взаємодіяти з зовнішніми системами.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу машинного перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, зверніть увагу, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується користуватися професійним перекладом, виконаним людиною. Ми не несемо відповідальності за будь-які непорозуміння чи неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
